In [15]:
%load_ext autoreload
    
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from src.phaseA import phaseA1, phaseA2, phaseA3
from src.phaseB import phaseB
import matplotlib.pyplot as plt

In [17]:
path_to_imgs = "data/scanned/*" #DENV_imgs/*"
scanned_path = "data/scanned/"
display= False

In [18]:
# Phase A.1 - Scanning images
Images = phaseA1(
    path_to_imgs, scanned_path,
    display=display, 
    do_white_balance=True,
    is_pre_scanned=True
)

# of Images to be analyzed: 7



In [19]:
# Phase A.2 - Grids
Grids = phaseA2(Images, display=display)

In [20]:
print(len(Grids))

7


In [21]:
# save test results
results = phaseB(Grids, display=display)

len(Grid_DS.get_blocks()): 7
len(Grid_DS.get_blocks()): 7
len(Grid_DS.get_blocks()): 7
len(Grid_DS.get_blocks()): 7
len(Grid_DS.get_blocks()): 7
len(Grid_DS.get_blocks()): 7
len(Grid_DS.get_blocks()): 7
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb
Error: Optimal parameters not found: The maximum number of function evaluations is exceeded.. Using mode_rgb


In [22]:
# Phase A.3 - Position Graph
graphs = phaseA3(Grids, display=display)

In [38]:
# --- Image Generation --- #
from src.generators.image_generation.RuleBasedGenerator import RuleBasedGenerator

RBG = RuleBasedGenerator(graphs, results)

starting_indexes = [(2,4)]

RBG.setup([])

[]
Folder /home/matheus.berbet001/AmpliVision/PhaseAB/data/generated_images/blank cleared.
unique_labels = dict_keys(['lung', 'thyroid', 'ovarian', 'prostate', 'skin', 'control', 'breast'])
label_mapping = {'breast': 0, 'control': 1, 'lung': 2, 'ovarian': 3, 'prostate': 4, 'skin': 5, 'thyroid': 6}
setup completed in 0.0 seconds


In [24]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import datasets, layers, models
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'


In [31]:
n = 20 # minumum 10

targets=["breast", "ovarian", "prostate"]
train_labels = [targets[i % len(targets)] for i in range(n * len(targets))]
test_labels = [targets[i % len(targets)] for i in range(int(n/10) * len(targets))]

print(train_labels)
print(test_labels)

['breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate']
['breast', 'ovarian', 'prostate', 'breast', 'ovarian', 'prostate']


In [33]:
# GENERATE AND SAVE A TON OF IMAGES
RBG.generate(batch_size=n, targets=train_labels, rotation=None, noise=0.05, rgb=True, save=True)

-------------------------------------------------- 
Rule Based Generator
 --------------------------------------------------


In [ ]:
model = models.Sequential()
model.add(tf.keras.Input(shape=(1204,1204, 3)))
model.add(layers.Conv2D(32, (16, 16), activation='relu', input_shape=(128, 128, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (16, 16), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(12, (32, 32), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(10))
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
from src.phaseA import ImageLoader


images = ImageLoader.load_images("data/generated_images/final/*")
print(len(images))

In [ ]:
training_split = .9

train_data = []
validation_data = []
train_labels = []
test_labels = []

# one hot encoding
label_mapping = {
    label: idx 
    for idx, label 
    in enumerate(sorted(set(targets)))
}

# training indexes
for i in range(len(targets)):
    t_start = (len(images)*i)//len(targets)
    t_end = int((len(images)*(i+1))//len(targets) * training_split)
    
    v_start = t_end
    v_end = (len(images)*(i+1))//len(targets)
    print(t_end - t_start , v_end- v_start)

    # img inputs
    train_data.extend(images[t_start:t_end])
    validation_data.extend(images[v_start:v_end])

    # encoded labels
    train_labels.extend([
        tf.keras.utils.to_categorical(
            label_mapping[targets[i]], num_classes=len(label_mapping)
        ) for j in range(t_start, t_end)
    ]
    )
    test_labels.extend([
        tf.keras.utils.to_categorical(
            label_mapping[targets[i]], num_classes=len(label_mapping)
        ) for j in range(v_start, v_end)
    ]
    )

print(len(train_data))
print(len(validation_data))


train_labels = np.array(train_labels)
test_labels = np.array(test_labels)
print(len(train_labels))
print(len(test_labels)

In [ ]:
import numpy as np

train_data = np.array(train_data)
validation_data = np.array(train_data)

train_data = train_data / 255.0
validation_data = validation_data / 255.0

In [ ]:


with tf.device('/GPU:0'):
#"""
    history = model.fit(
        train_data,
        train_labels,
        epochs=10, 
        validation_data= (
            validation_data,
            test_labels
        )
    )
#"""